<dl>
    <dt style="color:yellow;font-weight:500;"><a href="#concat">concat()</a></dt>
    <dd>Concatenate multiple objects (Series/DataFrame) along a particular axis</dd>
    <dt style="color:red;font-weight:500;"><a href="#merge">merge()</a></dt>
    <dd>Combine two objects (Series/DataFrame) with SQL-style joining</dd>
    <dt style="color:cyan;font-weight:500;"><a href="#join">join()</a></dt>
    <dd>Merge multiple DataFrames along the columns</dd>  
</dl>

In [58]:
import pandas as pd
import numpy as np

# <span id="concat" style="color:yellow;">concat()</span>

    pd.concat([df1, df2, ...],
            axis=0,
            join="outer",
            ignore_index=True,
            keys=["x", "y", "z",...])


In [59]:
df1 = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    },
    index=[0, 1, 2, 3],
)
df2 = pd.DataFrame(
    {
        "A": ["A4", "A5", "A6", "A7"],
        "B": ["B4", "B5", "B6", "B7"],
        "C": ["C4", "C5", "C6", "C7"],
        "D": ["D4", "D5", "D6", "D7"],
    },
    index=[4, 5, 6, 7],
)
df3 = pd.DataFrame(
    {
        "A": ["A8", "A9", "A10", "A11"],
        "B": ["B8", "B9", "B10", "B11"],
        "C": ["C8", "C9", "C10", "C11"],
        "D": ["D8", "D9", "D10", "D11"],
    },
    index=[8, 9, 10, 11],
)

In [60]:
# Collect all the DataFrame objects in a list before using concat()
frames = [df1, df2, df3]

# Concatenate the DataFrames along axis=0
# NOTE: axis=0 by default (concatenate vertically from top to bottom)
#       join="outer" by default (union of all axis values)
result = pd.concat(frames)

# Display the resulting DataFrame
result

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3
4,A4,B4,C4,D4
5,A5,B5,C5,D5
6,A6,B6,C6,D6
7,A7,B7,C7,D7
8,A8,B8,C8,D8
9,A9,B9,C9,D9


## Joining Logic of the Resulting Axis
Use join attribute to specify what to do with axis values that don't exist in the first DataFrame.

<span style="color:#00FF00;font-weight:500">join="outer"</span> : takes the union of all axis values.<br/>
<span style="color:#FF0000;font-weight:500">join="inner"</span> : takes the intersection of all axis values.

In [61]:
df4 = pd.DataFrame(
    {
        "B": ["B2", "B3", "B6", "B7"],
        "D": ["D2", "D3", "D6", "D7"],
        "F": ["F2", "F3", "F6", "F7"],
    },
    index=[2, 3, 6, 7],
)
result = pd.concat([df1, df4], axis=1)
result

,A,B,C,D,B,D,F
0,A0,B0,C0,D0,NaN,NaN,NaN
1,A1,B1,C1,D1,NaN,NaN,NaN
2,A2,B2,C2,D2,B2,D2,F2
3,A3,B3,C3,D3,B3,D3,F3
6,NaN,NaN,NaN,NaN,B6,D6,F6
7,NaN,NaN,NaN,NaN,B7,D7,F7


In [62]:
result = pd.concat([df1, df4], axis=1, join="inner")
result

,A,B,C,D,B,D,F
2,A2,B2,C2,D2,B2,D2,F2
3,A3,B3,C3,D3,B3,D3,F3


In [63]:
# An effective "left" join i.e. using the exact index specified.
# This is achieved by reindexing.
result = pd.concat([df1, df4], axis=1).reindex(df1.index)
result

,A,B,C,D,B,D,F
0,A0,B0,C0,D0,NaN,NaN,NaN
1,A1,B1,C1,D1,NaN,NaN,NaN
2,A2,B2,C2,D2,B2,D2,F2
3,A3,B3,C3,D3,B3,D3,F3


## Ignoring indexes on the concatenation axis

In [64]:
# Concatenate along axis=0 (by default) and
# ignore old indexes; the new index is normal 0-n values.
result = pd.concat([df1, df4], ignore_index=True, sort=False)
result

,A,B,C,D,F
0,A0,B0,C0,D0,NaN
1,A1,B1,C1,D1,NaN
2,A2,B2,C2,D2,NaN
3,A3,B3,C3,D3,NaN
4,NaN,B2,NaN,D2,F2
5,NaN,B3,NaN,D3,F3
6,NaN,B6,NaN,D6,F6
7,NaN,B7,NaN,D7,F7


## Concatenating Series and DataFrame together

In [65]:
# NOTE: The name of the Series (X) becomes the name of the column.
s1 = pd.Series(["X0", "X1", "X2", "X3"], name="X")
result = pd.concat([df1, s1], axis=1)
result

,A,B,C,D,X
0,A0,B0,C0,D0,X0
1,A1,B1,C1,D1,X1
2,A2,B2,C2,D2,X2
3,A3,B3,C3,D3,X3


In [66]:
# NOTE: If the Series is unnamed, the columns will be numbered consecutively
s2 = pd.Series(["_0", "_1", "_2", "_3"])
result = pd.concat([df1, s2, s2, s2], axis=1)
result

,A,B,C,D,0,1,2
0,A0,B0,C0,D0,_0,_0,_0
1,A1,B1,C1,D1,_1,_1,_1
2,A2,B2,C2,D2,_2,_2,_2
3,A3,B3,C3,D3,_3,_3,_3


In [67]:
# ignore_index=True, drops all name references (replace names of all columns)
result = pd.concat([df1, s1], axis=1, ignore_index=True)
result

,0,1,2,3,4
0,A0,B0,C0,D0,X0
1,A1,B1,C1,D1,X1
2,A2,B2,C2,D2,X2
3,A3,B3,C3,D3,X3


## Resulting Keys

In [68]:
# Each key is associated with each original DataFrame, creating a MultiIndex
result = pd.concat(frames, keys=["x", "y", "z"])
result

A    B    C    D
x 0    A0   B0   C0   D0
  1    A1   B1   C1   D1
  2    A2   B2   C2   D2
  3    A3   B3   C3   D3
y 4    A4   B4   C4   D4
  5    A5   B5   C5   D5
  6    A6   B6   C6   D6
  7    A7   B7   C7   D7
z 8    A8   B8   C8   D8
  9    A9   B9   C9   D9
  10  A10  B10  C10  D10
  11  A11  B11  C11  D11

In [69]:
s3 = pd.Series([0, 1, 2, 3], name="foo")
s4 = pd.Series([0, 1, 2, 3])
s5 = pd.Series([0, 1, 4, 5])

pd.concat([s3, s4, s5], axis=1)

,foo,0,1
0,0,0,0
1,1,1,1
2,2,2,4
3,3,3,5


In [70]:
# Using keys, overrides the old column names
pd.concat([s3, s4, s5], axis=1, keys=["red", "blue", "yellow"])

,red,blue,yellow
0,0,0,0
1,1,1,1
2,2,2,4
3,3,3,5


In [71]:
# Passing a dict, the dict's keys will be used as the arguments for keys.
pieces = {"x": df1, "y": df2, "z": df3}
result = pd.concat(pieces)
result

A    B    C    D
x 0    A0   B0   C0   D0
  1    A1   B1   C1   D1
  2    A2   B2   C2   D2
  3    A3   B3   C3   D3
y 4    A4   B4   C4   D4
  5    A5   B5   C5   D5
  6    A6   B6   C6   D6
  7    A7   B7   C7   D7
z 8    A8   B8   C8   D8
  9    A9   B9   C9   D9
  10  A10  B10  C10  D10
  11  A11  B11  C11  D11

In [72]:
result = pd.concat(pieces, keys=["z", "y"])
result

A    B    C    D
z 8    A8   B8   C8   D8
  9    A9   B9   C9   D9
  10  A10  B10  C10  D10
  11  A11  B11  C11  D11
y 4    A4   B4   C4   D4
  5    A5   B5   C5   D5
  6    A6   B6   C6   D6
  7    A7   B7   C7   D7

In [73]:
result.index.levels

FrozenList([['z', 'y'], [4, 5, 6, 7, 8, 9, 10, 11]])

In [74]:
result = pd.concat(
    pieces,
    keys=["x", "y", "z"],
    levels=[["z", "y", "x", "w"]],
    names=["group_key"],
)
result

A    B    C    D
group_key                       
x         0    A0   B0   C0   D0
          1    A1   B1   C1   D1
          2    A2   B2   C2   D2
          3    A3   B3   C3   D3
y         4    A4   B4   C4   D4
          5    A5   B5   C5   D5
          6    A6   B6   C6   D6
          7    A7   B7   C7   D7
z         8    A8   B8   C8   D8
          9    A9   B9   C9   D9
          10  A10  B10  C10  D10
          11  A11  B11  C11  D11

## Appending rows to a DataFrame

In [75]:
# Append a Series as a single row to a DataFrame:
#   - Convert the Series to a DataFrame and
#   - Use concat()
s2 = pd.Series(["X0", "X1", "X2", "X3"], index=["A", "B", "C", "D"])

In [76]:
result = pd.concat([df1, s2.to_frame().T], ignore_index=True)
result

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3
4,X0,X1,X2,X3


# <span id="merge" style="color:red;">merge()</span>
Performs join operations similar to relational SQL databases.

<span style="font-size:16pt;font-weight:500;color:brown;text-decoration:underline;">ATTRIBUTES:</span><br/>
<b>on</b> : label or list Column or index level names to join on. These must be found in both DataFrames.<br/> 
<b>how</b> : Type of merge to be performed.{'left', 'right', 'outer', 'inner', 'cross'}, default 'inner'<br/>

   
<span style="font-size:16pt;font-weight:500;color:brown;text-decoration:underline;">COMMON SQL-STYLE JOIN OPERATIONS:</span><br/>
<dl>
    <dt style="font-weight:500;">one-to-one</dt>
    <dd>Join two DataFrames on their indexes, which must contain unique values.</dd>
    <dt style="font-weight:500;">many-to-one</dt>
    <dd>Join a unique index to one or more columns of another DataFrame</dd>
    <dt style="font-weight:500;">many-to-many</dt>
    <dd>Join columns on columns</dd>
</dl>

In [77]:
left = pd.DataFrame(
    {
        "key": ["K0", "K1", "K2", "K3"],
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
    }
)
right = pd.DataFrame(
    {
        "key": ["K0", "K1", "K2", "K3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    }
)

# join on column with name "key"
result = pd.merge(left, right, on="key")
result

,key,A,B,C,D
0,K0,A0,B0,C0,D0
1,K1,A1,B1,C1,D1
2,K2,A2,B2,C2,D2
3,K3,A3,B3,C3,D3


In [78]:
left = pd.DataFrame(
    {
        "key1": ["K0", "K0", "K1", "K2"],
        "key2": ["K0", "K1", "K0", "K1"],
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
    }
)
right = pd.DataFrame(
    {
        "key1": ["K0", "K1", "K1", "K2"],
        "key2": ["K0", "K0", "K0", "K0"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    }
)

# how="left" : use only keys from left frame. In this case:
#              (K0, K0), (K0, K1), (K1, K0) and (K2, K1)
result = pd.merge(
    left,
    right,
    how="left",
    on=["key1", "key2"],
)
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K0,K1,A1,B1,NaN,NaN
2,K1,K0,A2,B2,C1,D1
3,K1,K0,A2,B2,C2,D2
4,K2,K1,A3,B3,NaN,NaN


In [79]:
# how="right" : use only keys from right frame.
result = pd.merge(left, right, how="right", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K1,K0,A2,B2,C1,D1
2,K1,K0,A2,B2,C2,D2
3,K2,K0,NaN,NaN,C3,D3


In [80]:
# how="outer" : Use union of keys from both frames
result = pd.merge(left, right, how="outer", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K0,K1,A1,B1,NaN,NaN
2,K1,K0,A2,B2,C1,D1
3,K1,K0,A2,B2,C2,D2
4,K2,K0,NaN,NaN,C3,D3
5,K2,K1,A3,B3,NaN,NaN


In [81]:
# how="inner" : use intersection of keys
result = pd.merge(left, right, how="inner", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K1,K0,A2,B2,C1,D1
2,K1,K0,A2,B2,C2,D2


In [82]:
# how="cross" : create the Cartesian product from both frames
result = pd.merge(left, right, how="cross")
result

,key1_x,key2_x,A,B,key1_y,key2_y,C,D
0,K0,K0,A0,B0,K0,K0,C0,D0
1,K0,K0,A0,B0,K1,K0,C1,D1
2,K0,K0,A0,B0,K1,K0,C2,D2
3,K0,K0,A0,B0,K2,K0,C3,D3
4,K0,K1,A1,B1,K0,K0,C0,D0
5,K0,K1,A1,B1,K1,K0,C1,D1
6,K0,K1,A1,B1,K1,K0,C2,D2
7,K0,K1,A1,B1,K2,K0,C3,D3
8,K1,K0,A2,B2,K0,K0,C0,D0
9,K1,K0,A2,B2,K1,K0,C1,D1


In [83]:
df = pd.DataFrame(
    {
        "Let": ["A", "B", "C"],
        "Num": [1, 2, 3],
    },
)
df

,Let,Num
0,A,1
1,B,2
2,C,3


In [84]:
ser = pd.Series(
    ["a", "b", "c", "d", "e", "f"],
    index=pd.MultiIndex.from_arrays([["A", "B", "C"] * 2, [1, 2, 3, 4, 5, 6]], names=["Let", "Num"]),
)
ser.reset_index()

,Let,Num,0
0,A,1,a
1,B,2,b
2,C,3,c
3,A,4,d
4,B,5,e
5,C,6,f


In [85]:
# NOTE: the default how value is "inner" (intersection of keys)
pd.merge(df, ser.reset_index(), on=["Let", "Num"])

,Let,Num,0
0,A,1,a
1,B,2,b
2,C,3,c


In [86]:
# Perform an outer join
left = pd.DataFrame({"A": [1, 2], "B": [2, 2]})
right = pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]})
result = pd.merge(left, right, on="B", how="outer")
result

,A_x,B,A_y
0,1,2,4
1,1,2,5
2,1,2,6
3,2,2,4
4,2,2,5
5,2,2,6


## Merge Key Uniqueness

In [87]:
# Check the uniqueness of merge keys
left = pd.DataFrame({"A": [1, 2], "B": [2, 2]})
right = pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]})

# This raises an error, since merge keys are not unique
# result = pd.merge(left, right, on="B", how="outer", validate="one_to_one")

## Merge Result Indicator

In [88]:
df1 = pd.DataFrame({"col1": [0, 1], "col_left": ["a", "b"]})
df2 = pd.DataFrame({"col1": [1, 2, 2], "col_right": [2, 2, 2]})
pd.merge(df1, df2, on="col1", how="outer", indicator=True)

,col1,col_left,col_right,_merge
0,0,a,NaN,left_only
1,1,b,2.0,both
2,2,NaN,2.0,right_only
3,2,NaN,2.0,right_only


## Overlapping Value Columns

In [89]:
left = pd.DataFrame({"k": ["K0", "K1", "K2"], "v": [1, 2, 3]})
right = pd.DataFrame({"k": ["K0", "K0", "K3"], "v": [4, 5, 6]})

print(left)
print()
print(right)

    k  v
0  K0  1
1  K1  2
2  K2  3

    k  v
0  K0  4
1  K0  5
2  K3  6


In [90]:
# by default how="inner"
result = pd.merge(left, right, on="k")
result

,k,v_x,v_y
0,K0,1,4
1,K0,1,5


In [91]:
# Above we have overlapping column names (v in both frames).
# To make clear where the result columns come from:
result = pd.merge(left, right, on="k", suffixes=("_l", "_r"))
result

,k,v_l,v_r
0,K0,1,4
1,K0,1,5


# <span id="join" style="color:cyan;">join()</span>
Combines the columns of multiple (potentially differently-indexed) DataFrames, into a <b>single</b> DataFrame.

In [92]:
left = pd.DataFrame({"A": ["A0", "A1", "A2"], "B": ["B0", "B1", "B2"]}, index=["K0", "K1", "K2"])
right = pd.DataFrame({"C": ["C0", "C2", "C3"], "D": ["D0", "D2", "D3"]}, index=["K0", "K2", "K3"])

# NOTE: by default how="left" and if on is not present, joins index-on-index
result = left.join(right)
result

,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2


In [93]:
result = left.join(right, how="outer")
result

,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2
K3,NaN,NaN,C3,D3


In [94]:
result = left.join(right, how="inner")
result

,A,B,C,D
K0,A0,B0,C0,D0
K2,A2,B2,C2,D2


In [95]:
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key": ["K0", "K1", "K0", "K1"],
    }
)
right = pd.DataFrame(
    {
        "C": ["C0", "C1"],
        "D": ["D0", "D1"],
    },
    index=["K0", "K1"],
)

result = left.join(right, on="key")
result

,A,B,key,C,D
0,A0,B0,K0,C0,D0
1,A1,B1,K1,C1,D1
2,A2,B2,K0,C0,D0
3,A3,B3,K1,C1,D1


In [96]:
result = pd.merge(left, right, left_on="key", right_index=True, how="left", sort=False)
result

,A,B,key,C,D
0,A0,B0,K0,C0,D0
1,A1,B1,K1,C1,D1
2,A2,B2,K0,C0,D0
3,A3,B3,K1,C1,D1


## Joining a Single Index to a MultiIndex

In [97]:
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2"],
        "B": ["B0", "B1", "B2"],
    },
    index=pd.Index(
        ["K0", "K1", "K2"],
        name="key",
    ),
)

right = pd.DataFrame(
    {
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    },
    index=pd.MultiIndex.from_tuples(
        [
            ("K0", "Y0"),
            ("K1", "Y1"),
            ("K2", "Y2"),
            ("K2", "Y3"),
        ],
        names=["key", "Y"],
    ),
)

In [98]:
print(left)
print(right)

      A   B
key        
K0   A0  B0
K1   A1  B1
K2   A2  B2
         C   D
key Y         
K0  Y0  C0  D0
K1  Y1  C1  D1
K2  Y2  C2  D2
    Y3  C3  D3


In [99]:
result = left.join(right, how="inner")
result

A   B   C   D
key Y                 
K0  Y0  A0  B0  C0  D0
K1  Y1  A1  B1  C1  D1
K2  Y2  A2  B2  C2  D2
    Y3  A2  B2  C3  D3

## Joining with two MultiIndex

In [100]:
left = pd.DataFrame(
    {"v1": range(12)},
    index=pd.MultiIndex.from_product(
        [list("abc"), list("xy"), [1, 2]],
        names=["abc", "xy", "num"],
    ),
)
left

v1
abc xy num    
a   x  1     0
       2     1
    y  1     2
       2     3
b   x  1     4
       2     5
    y  1     6
       2     7
c   x  1     8
       2     9
    y  1    10
       2    11

In [101]:
right = pd.DataFrame(
    {"v2": [100 * i for i in range(1, 7)]},
    index=pd.MultiIndex.from_product([list("abc"), list("xy")], names=["abc", "xy"]),
)

right

v2
abc xy     
a   x   100
    y   200
b   x   300
    y   400
c   x   500
    y   600

In [102]:
left.join(right, on=["abc", "xy"], how="inner")

v1   v2
abc xy num         
a   x  1     0  100
       2     1  100
    y  1     2  200
       2     3  200
b   x  1     4  300
       2     5  300
    y  1     6  400
       2     7  400
c   x  1     8  500
       2     9  500
    y  1    10  600
       2    11  600

## Merging on a combination of columns and index levels

In [103]:
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key2": ["K0", "K1", "K0", "K1"],
    },
    index=pd.Index(
        ["K0", "K0", "K1", "K2"],
        name="key1",
    ),
)

In [104]:
right = pd.DataFrame(
    {"C": ["C0", "C1", "C2", "C3"], "D": ["D0", "D1", "D2", "D3"], "key2": ["K0", "K0", "K0", "K1"]},
    index=pd.Index(
        ["K0", "K1", "K2", "K2"],
        name="key1",
    ),
)

In [105]:
print(left)
print()
print(right)

       A   B key2
key1             
K0    A0  B0   K0
K0    A1  B1   K1
K1    A2  B2   K0
K2    A3  B3   K1

       C   D key2
key1             
K0    C0  D0   K0
K1    C1  D1   K0
K2    C2  D2   K0
K2    C3  D3   K1


In [106]:
# Default how="inner" i.e. intersection of values
result = left.merge(right, on=["key1", "key2"])
result

,A,B,key2,C,D
key1,,,,,
K0,A0,B0,K0,C0,D0
K1,A2,B2,K0,C1,D1
K2,A3,B3,K1,C3,D3


## Joining Multiple DataFrame

In [107]:
# Join multiple frames on their indexes
# Default how="left" i.e. use only values from left frame
right2 = pd.DataFrame({"v": [7, 8, 9]}, index=["K1", "K1", "K2"])
result = left.join([right, right2])
result

,A,B,key2_x,C,D,key2_y,v
key1,,,,,,,
K0,A0,B0,K0,C0,D0,K0,NaN
K0,A1,B1,K1,C0,D0,K0,NaN
K1,A2,B2,K0,C1,D1,K0,7.0
K1,A2,B2,K0,C1,D1,K0,8.0
K2,A3,B3,K1,C2,D2,K0,9.0
K2,A3,B3,K1,C3,D3,K1,9.0


## combine_first()
Updates missing values in one DataFrame, with non-missing values in another

In [108]:
df1 = pd.DataFrame(
    [
        [np.nan, 3.0, 5.0],
        [-4.6, np.nan, np.nan],
        [np.nan, 7.0, np.nan],
    ]
)
df2 = pd.DataFrame(
    [
        [-42.6, np.nan, -8.2],
        [-5.0, 1.6, 4],
    ],
    index=[1, 2],
)

In [109]:
result = df1.combine_first(df2)
result

,0,1,2
0,NaN,3.0,5.0
1,-4.6,NaN,-8.2
2,-5.0,7.0,4.0


# merge_ordered()
Combines order data (numeric or time series) with optional filling of missing data.

In [110]:
left = pd.DataFrame(
    {
        "k": ["K0", "K1", "K1", "K2"],
        "lv": [1, 2, 3, 4],
        "s": ["a", "b", "c", "d"],
    }
)
right = pd.DataFrame(
    {
        "k": ["K1", "K2", "K4"],
        "rv": [1, 2, 3],
    }
)

In [111]:
print(left)
print(right)

    k  lv  s
0  K0   1  a
1  K1   2  b
2  K1   3  c
3  K2   4  d
    k  rv
0  K1   1
1  K2   2
2  K4   3


In [112]:
pd.merge_ordered(left, right, fill_method="ffill", left_by="s")

,k,lv,s,rv
0,K0,1.0,a,NaN
1,K1,1.0,a,1.0
2,K2,1.0,a,2.0
3,K4,1.0,a,3.0
4,K1,2.0,b,1.0
5,K2,2.0,b,2.0
6,K4,2.0,b,3.0
7,K1,3.0,c,1.0
8,K2,3.0,c,2.0
9,K4,3.0,c,3.0
